# 🧹 02-机器学习 · 推导笔记 13：特征工程与模型评估（手写全流程）

> 面试里“特征工程 + 评估”是**必考但最容易被忽视**的板块：AUC 怎么算、KFold 怎么切、为什么类别不平衡要调阈值——都是现场手写题。

> 💡 一句话类比：**特征工程是“把食材处理好再下锅”**——土豆不削皮、肉不切块，再好的菜谱（模型）也白搭；**模型评估是“上桌前先试吃”**——不能只问“熟了没”（准确率），还要看咸淡、分量、挑食的人爱不爱吃（精确率/召回率/AUC）。

**运行环境**：Python 3.8+，仅依赖 numpy（对照 cell 需要 sklearn）。

机器学习标准流水线

> 📊 图 7：机器学习标准流水线——从原始数据到评估的一整条链路；评估不合格要“退回去”迭代（交叉验证闭环），且缩放统计量只在训练集上 fit。

## 📖 本章导读

> **这一章讲什么**（全书第 14 章）：决定上限的两件事——**特征工程**（缺失值/编码/缩放/离散化/特征选择）与**模型评估**（混淆矩阵 → P/R/F1 → **AUC 秩和推导** → 交叉验证/学习曲线），最后是类别不平衡五板斧。

**学完本章，你应该能**：
1. 列出流水线各环节与每步的注意点（树模型不用缩放等）；
2. 解释**数据泄漏**为什么可怕（评估虚高）、怎么避免（统计量只在训练集 fit）；
3. 手算 AUC：$AUC=P(score_+>score_-)=\frac{R_+-n_+(n_++1)/2}{n_+n_-}$；
4. 讲清类别不平衡的评估（AUC/PR）与处理（五板斧）。

> 📌 全书第 14 章（终章）。阅读路径：本章导读 → 特征工程 · 评估指标 · 交叉验证 → 自测清单 → 本章小结。

## §1 缺失值处理

| 方法 | 做法 | 适用 | 风险 |
|------|------|------|------|
| 删除 | 删行/删列 | 缺失极少 | 删太多丢信息 |
| 统计量填充 | 均值/中位数/众数 | 数值/分类 | 扭曲分布、降低方差 |
| 插值 | 线性/样条/前向填充 | 时间序列 | 不适合突变数据 |
| 模型预测 | 用其它特征预测缺失列 | 特征间相关性强 | 计算贵、有泄漏风险 |
| 标记法 | 加“是否缺失”指示列 | 缺失本身有含义 | 维度增加 |

> 💡 通俗例子：问卷调查里“收入”很多人不填。若直接删行，月薪 3 万的人可能更不愿填——删掉后样本系统性偏向低收入，模型就学歪了。**先看缺失是否随机（MCAR/MAR/MNAR），再选方法**，是面试加分回答。

In [ ]:
import numpy as np

# 示例：10 个样本 2 个特征，部分缺失（NaN）
X = np.array([[1.0, np.nan], [2.0, 3.0], [np.nan, 5.0], [4.0, 6.0], [5.0, np.nan]])
print("原始数据（NaN=缺失）：")
print(X)

# 均值填充（手写）
def mean_fill(X):
    Xc = X.copy()
    for j in range(X.shape[1]):
        col = Xc[:, j]
        m = np.nanmean(col)
        Xc[np.isnan(col), j] = m
    return Xc

print("\n均值填充后：")
print(mean_fill(X))

# 缺失指示列：把"缺失与否"本身当特征
miss = np.isnan(X).astype(int)
print("\n缺失指示矩阵（可拼接为额外特征）：")
print(miss)

## §2 分类特征编码

| 编码 | 做法 | 例子 | 注意 |
|------|------|------|------|
| Label | 类别 → 0,1,2… | 红=0 绿=1 蓝=2 | **隐含大小关系**，无序类别慎用 |
| One-Hot | 每个类别一列 0/1 | 红=100 绿=010 蓝=001 | 维度爆炸，可配哈希/嵌入 |
| Ordinal | 有序类别 → 整数 | 低中高=0,1,2 | 用于有顺序的类别（学历/等级） |
| Target | 用类别对应的目标均值编码 | 颜色→该色点击率 | 需交叉验证防泄漏，易过拟合 |

> 💡 通俗例子：给“交通方式”编码——地铁、公交、步行**没有大小关系**，用 Label 编码会凭空造出“地铁 > 公交 > 步行”的顺序，模型会学出错误的单调关系；应该 One-Hot。而“学历”**有顺序**，Ordinal 编码更合适。

In [ ]:
# 手写 Label + One-Hot 编码
colors = np.array(["红", "绿", "蓝", "红", "绿"])
uniq = np.unique(colors)
label = np.array([np.where(uniq == c)[0][0] for c in colors])
onehot = np.eye(len(uniq), dtype=int)[label]

print("唯一类别:", uniq)
print("Label 编码:", label)
print("One-Hot 编码:")
print(onehot)

## §3 特征缩放：归一化 vs 标准化

**Min-Max 归一化**（把数据压到 $[0,1]$）：

$$x' = \frac{x - \min(x)}{\max(x) - \min(x)}$$

**z-score 标准化**（均值 0、方差 1）：

$$x' = \frac{x - \mu}{\sigma}$$

**什么时候必须缩放**：

- **距离类**（KNN、SVM-RBF、K-Means、PCA）：量纲主导距离，必做；
- **梯度类**（LR、神经网络）：不缩放收敛慢、量纲大的特征梯度被放大；
- **树模型**（决策树/GBDT/RF）：按阈值切分，**单调变换不影响**，可不做。

⚠️ 缩放统计量（$\mu, \sigma, \min, \max$）**只能在训练集上算**，再应用到验证/测试集——否则测试信息泄漏进训练。

In [ ]:
def zscore(X):
    mu = X.mean(axis=0)
    sd = X.std(axis=0)
    sd[sd == 0] = 1.0
    return (X - mu) / sd, mu, sd

def minmax(X):
    lo, hi = X.min(axis=0), X.max(axis=0)
    return (X - lo) / (hi - lo), lo, hi

X = np.array([[0.0, 100.0], [1.0, 200.0], [2.0, 300.0], [3.0, 400.0]])
Xs, mu, sd = zscore(X)
Xm, lo, hi = minmax(X)
print("原始："); print(X)
print("\nz-score（列均值≈0，方差≈1）："); print(Xs.round(3))
print("\nmin-max（列范围[0,1]）："); print(Xm.round(3))

# 测试集用训练集统计量（防泄漏）
X_test = np.array([[4.0, 500.0]])
print("\n测试集用训练集统计量标准化：", ((X_test - mu) / sd).round(3).ravel())

## §4 离散化与特征交叉

**离散化**：连续值 → 分箱（等宽/等频/聚类）。例子：年龄 18–25/26–35/36–50/50+。好处：对异常值鲁棒、表达非线性、配合 One-Hot 与 LR 更好用。

**特征交叉**：把两个（或多个）特征组合成新特征。例子：**年龄 × 收入**、**城市 × 品类**——“一线城市 × 数码”和“三线城市 × 数码”的购买力完全不同，交叉后模型才能学到这种**组合效应**。

> 💡 经典工业实践：GBDT+LR——GBDT 自动做特征交叉（叶子节点 = 高阶交叉特征的 one-hot），把叶子输出喂给 LR，LR 学权重。这就是推导笔记 6 的延伸应用。

In [ ]:
# 手写等宽分箱 + 交叉特征
ages = np.array([19, 23, 31, 29, 42, 55, 37, 60])
income = np.array([5, 8, 12, 10, 20, 30, 18, 40])  # 万元/年

# 等宽分箱：3 箱
bins = np.linspace(ages.min(), ages.max(), 4)
age_bin = np.digitize(ages, bins[1:-1])
print("年龄分箱(0/1/2):", age_bin)

# 交叉特征：年龄箱 × 收入箱（组合索引）
inc_bin = np.digitize(income, np.linspace(income.min(), income.max(), 4)[1:-1])
cross = age_bin * 10 + inc_bin   # 把二维组合压成一维 ID
print("收入分箱(0/1/2):", inc_bin)
print("交叉特征 ID (年龄箱*10+收入箱):", cross)

## §5 特征选择：过滤 / 包裹 / 嵌入

| 方法 | 思路 | 代表 | 特点 |
|------|------|------|------|
| 过滤式 | 先按统计量筛，再训练 | 方差/卡方/互信息 | 快、与模型无关 |
| 包裹式 | 用模型效果评估特征子集 | 递归特征消除 RFE | 准、计算贵 |
| 嵌入式 | 训练时自动选 | L1 稀疏、树重要性 | 折中，实践中常用 |

**为什么要做**：降维度、防过拟合、提解释性、省算力。

> 💡 通俗例子：筛简历——过滤式 = 先按硬门槛（学历/年限）批量筛掉大半；包裹式 = 逐个试“加上这个人对团队产出影响多大”；嵌入式 = 边干活边淘汰（试用期观察）。

In [ ]:
# 手写方差过滤：剔除几乎不变的噪声特征
rng = np.random.default_rng(0)
X = np.hstack([
    rng.normal(0, 1, (200, 3)),   # 3 个有信息的特征
    np.full((200, 2), 0.5),       # 2 个常数特征（方差=0，可剔除）
    rng.normal(0, 0.01, (200, 2)),# 2 个近常数噪声
])

var = X.var(axis=0)
keep = np.where(var > 0.05)[0]
print("各特征方差:", var.round(4))
print(f"方差阈值 0.05 后保留特征: {keep}  (常数/近常数特征被剔除)")

# sklearn 对照
from sklearn.feature_selection import VarianceThreshold
sel = VarianceThreshold(threshold=0.05).fit(X)
print(f"sklearn VarianceThreshold 保留: {sel.get_support(indices=True)}")

## §6 评估指标：分类与排序（AUC 手写）

**分类指标**（以正类为“患病/点击/欺诈”）：

$$\text{Precision} = \frac{TP}{TP+FP}, \qquad \text{Recall} = \frac{TP}{TP+FN}, \qquad F1 = \frac{2 \cdot P \cdot R}{P + R}$$

- **准确率**：整体对的比例——**类别不平衡时严重失真**（99% 负样本，全猜负类也 99%）；
- **精确率**：预测为正的里面有多少真对（“别冤枉好人”）；
- **召回率**：真正的正类被找出多少（“别放过坏人”）；
- **F1**：两者调和平均（对小数敏感，比算术平均更严格）。

**AUC（ROC 曲线下面积）**：随机抽一个正样本、一个负样本，**正样本得分高于负样本的概率**。取值范围 $[0.5, 1]$（0.5=瞎猜，1=完美排序），**与阈值无关**、对类别不平衡不敏感——所以排序/风控场景看 AUC 而不是准确率。

**计算**：按得分排序后用秩和（Mann-Whitney U）：

$$\text{AUC} = \frac{\sum r_i - n_+(n_+ + 1)/2}{n_+ n_-}$$

其中 $r_i$ 是正样本的秩（排名）。

In [ ]:
def auc_hand(y, score):
    """手写 AUC：秩和公式"""
    n = len(y)
    order = np.argsort(score)
    ranks = np.empty(n)
    ranks[order] = np.arange(1, n + 1)
    n_pos = int(y.sum())
    n_neg = n - n_pos
    rank_sum_pos = ranks[y == 1].sum()
    return (rank_sum_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

rng = np.random.default_rng(1)
y = rng.integers(0, 2, 200)                      # 200 个 0/1 标签
score = 0.8 * y + rng.normal(0, 0.5, 200)       # 得分与标签正相关

print(f"手写 AUC = {auc_hand(y, score):.4f}")

from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
print(f"sklearn AUC = {roc_auc_score(y, score):.4f}   (应完全一致)")

# 分类指标手算 vs sklearn
pred = (score > np.median(score)).astype(int)
TP = ((pred == 1) & (y == 1)).sum(); FP = ((pred == 1) & (y == 0)).sum()
FN = ((pred == 0) & (y == 1)).sum()
P = TP / (TP + FP); R = TP / (TP + FN); F1 = 2 * P * R / (P + R)
print(f"手写 Precision={P:.3f} Recall={R:.3f} F1={F1:.3f}")
ps, rs, fs, _ = precision_recall_fscore_support(y, pred, average="binary")
print(f"sklearn P={ps:.3f} R={rs:.3f} F1={fs:.3f}")

In [ ]:
# ---- 可视化：ROC 曲线与 AUC 面积（用 §6 的数据，运行本 cell 生成）----
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def roc_points(y, score):
    """按分数降序累计 TPR/FPR（等价于画 ROC 的点）"""
    order = np.argsort(-score)
    ys = y[order]
    tpr = np.cumsum(ys) / max(ys.sum(), 1e-9)
    fpr = np.cumsum(1 - ys) / max((1 - ys).sum(), 1e-9)
    return np.r_[0.0, fpr, 1.0], np.r_[0.0, tpr, 1.0]

fpr, tpr = roc_points(y, score)
plt.figure(figsize=(6, 5.5))
plt.plot(fpr, tpr, "b-", lw=2, label=f"AUC = {auc_hand(y, score):.4f}")
plt.plot([0, 1], [0, 1], "k--", lw=1.2, label="随机猜测 (AUC=0.5)")
plt.fill_between(fpr, tpr, color="blue", alpha=0.12)
plt.xlabel("FPR（假正率）")
plt.ylabel("TPR（召回率）")
plt.title("ROC 曲线：AUC = 曲线下面积")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## §7 类别不平衡与交叉验证

**类别不平衡怎么办（面试五板斧）**：

1. **数据层面**：过采样（SMOTE 插值生成少数类）、欠采样（随机/编辑最近邻）；
2. **算法层面**：代价敏感（少数类误分惩罚大）、加权损失；
3. **评估层面**：用 AUC/PR/召回率，别只看准确率；
4. **决策层面**：**阈值移动**——模型输出的是概率，把阈值从 0.5 调低/调高，按业务代价选最优阈值；
5. 工业排序：用 **Pairwise/Listwise** 损失，绕开绝对概率偏差。

**为什么不能直接大量上采样**：简单复制少数类样本 → 模型对复制样本“背答案”，过拟合且方差失真；SMOTE 是在少数类样本之间**插值**生成新样本，缓解这个问题。

**交叉验证**：把数据切成 k 份，轮流拿 1 份当验证集、其余当训练集。**时间序列不能随机切**（未来信息泄漏），必须按时间顺序切。

In [ ]:
# 手写 K 折交叉验证切分
def kfold_indices(n, k, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)
    folds = np.array_split(idx, k)
    for i in range(k):
        test = folds[i]
        train = np.concatenate([folds[j] for j in range(k) if j != i])
        yield train, test

n = 12
for i, (tr, te) in enumerate(kfold_indices(n, 3)):
    print(f"折 {i}: 训练 {len(tr)} 个 / 验证 {len(te)} 个  (不重叠, 并集=全部 {n} 个)")

# sklearn 对照
from sklearn.model_selection import KFold
skf = KFold(n_splits=3, shuffle=True, random_state=0)
splits = list(skf.split(np.zeros(n)))
ok = all(len(te) == len(splits[0][1]) for _, te in splits)
print(f"\nsklearn KFold 每折验证集大小一致: {ok}")

# 时间序列切分：只允许 训练块时间 < 验证块时间
t = np.arange(100)
train_t = t[:80]; test_t = t[80:]
print(f"时间序列前向切分: 训练 0-79 / 验证 80-99  (避免未来泄漏)")

## §8 学习曲线与过拟合诊断

- **训练误差低、验证误差高** → 过拟合：加正则、减特征、加数据、早停；
- **两者都高** → 欠拟合：加特征、换强模型、减正则；
- **学习曲线**（横轴训练量，纵轴误差）：训练曲线与验证曲线**间距大** = 过拟合；**都很高且靠拢** = 欠拟合。

> 💡 通俗例子：考试（训练集）98 分、模考（验证集）60 分 = 背题过拟合；两样都 60 = 压根没学会。**看两条曲线的间距和高度**就能诊断。

**调参三板斧**：网格搜索（暴力枚举）、随机搜索（采样更快）、贝叶斯优化（用历史评估引导）。配合 K 折交叉验证评分，防调参过拟合（测试集只能碰一次）。

## §9 常见 bug 与边界（面试踩坑区）

- **缩放/填充/编码统计量用了全量数据**：必须只从训练集学，再 transform 测试集——否则数据泄漏（§3 已强调）；
- **One-Hot 后忘删原始列**：原始“颜色”和“红绿蓝”三列同时进模型，特征冗余且泄漏；
- **AUC 只排序不校准**：AUC 高不代表概率校准好（线上要概率就再评估对数损失）；
- **交叉验证里做特征选择**：特征选择要放进每一折的训练内部做，否则选择过程本身也泄漏；
- **类别不平衡只调数据不调阈值**：数据 + 阈值 + 代价三管齐下才是完整方案；
- **测试集反复调参**：测试集只能最终用一次，调参用验证集/交叉验证。

## §10 自测清单（L1 必会 / L2 进阶 / L3 挑战）

### L1 必会（10 问）

- [ ] 1. 缺失值处理的五种方法与取舍
- [ ] 2. Label vs One-Hot vs Ordinal 的适用场景
- [ ] 3. 归一化与标准化的公式与区别
- [ ] 4. 哪些模型必须缩放、哪些不必（树 vs 距离/梯度）
- [ ] 5. 准确率/精确率/召回率/F1 各自的含义
- [ ] 6. AUC 的物理意义（正样本排在负样本前的概率）
- [ ] 7. 类别不平衡时为什么不能只看准确率
- [ ] 8. K 折交叉验证流程与为什么时间序列不能随机切
- [ ] 9. 过拟合/欠拟合在学习曲线上怎么区分
- [ ] 10. 特征选择三种方式的区别

### L2 进阶（8 问）

- [ ] 11. 手写 AUC（秩和公式）
- [ ] 12. 手写 KFold 切分
- [ ] 13. SMOTE 的原理与为什么优于简单复制
- [ ] 14. 阈值移动怎么做（按业务代价选阈值）
- [ ] 15. 缩放统计量泄漏的完整故事（fit/transform 分离）
- [ ] 16. 交叉验证里特征选择为什么必须在折内做
- [ ] 17. 对数损失与 AUC 的互补性
- [ ] 18. 网格/随机/贝叶斯调参的取舍

### L3 挑战（4 问）

- [ ] 19. 手写 PR 曲线与 AP 计算
- [ ] 20. 嵌套交叉验证（模型选择 + 泛化估计分离）
- [ ] 21. 在线学习场景下特征漂移的检测（PSI）
- [ ] 22. 设计一个特征工程的完整流水线（清洗→编码→缩放→选择→交叉）

### 自测要点

- 10 分钟内盲写 AUC 与 KFold
- 能完整讲出“类别不平衡五板斧”
- 能指出数据泄漏的三个常见来源（缩放、填充、特征选择）

## 附录：参考与结课

- 《机器学习》周志华 第 2 章（模型评估与选择）
- 《统计学习方法》李航 第 1 章（经验风险/结构风险）
- sklearn 文档：`feature_selection` / `model_selection` / `metrics`

### 🎉 02-机器学习 推导笔记系列（13 篇）至此全部构成

> 💡 本笔记验收：`02-机器学习/README.md`「推导笔记」特征工程与评估篇打勾。

## 🏁 本章小结

- **原理一句话**：**特征决定上限，模型逼近上限**；评估要防泄漏、看排序指标。
- **必会公式**：AUC $=P(score_+>score_-)=\frac{R_+-n_+(n_++1)/2}{n_+n_-}$（秩和手算）。
- **面试怎么考**：流水线每步注意点；数据泄漏三场景（缩放/填充/SMOTE 只在训练集 fit）；AUC 为什么对类别不平衡不敏感；时间序列为什么不能随机交叉验证；类别不平衡五板斧。
- **易错点**：泄漏导致评估虚高；类别不平衡只看准确率；F1/AUC 混用。

> 🎉 全书最后一章完成：回到 00-总览 过一遍公式速查表与对比总表，再进 📝 `高频面试题.md` 逐条打勾。